# Victorian Wholesale Electricity Market Analysis

This notebook explores five-minute Victorian NEM wholesale price behaviour from **1 August 2025 to 31 July 2026**.

It focuses on:

- negative-price conditions
- demand and renewable-generation interactions
- solar and wind patterns
- intraday behaviour
- extreme price risk
- the 8 July 2026 extreme-price event
- weekday/weekend and monthly patterns

The Power BI report is the polished presentation layer; this notebook documents the analytical logic behind the findings.

## 1. Load the final analytical dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_FILE = Path("../data/final_market_dataset.csv")

final_data = pd.read_csv(DATA_FILE)
final_data["SETTLEMENTDATE"] = pd.to_datetime(
    final_data["SETTLEMENTDATE"]
)

print(final_data.shape)
print(final_data["SETTLEMENTDATE"].min())
print(final_data["SETTLEMENTDATE"].max())

## 2. Price overview

In [ ]:
price_summary = final_data["RRP"].agg(
    ["mean", "median", "min", "max"]
)

p95 = final_data["RRP"].quantile(0.95)
p99 = final_data["RRP"].quantile(0.99)

print(price_summary)
print(f"P95: {p95:.2f}")
print(f"P99: {p99:.2f}")

## 3. Negative price frequency

In [ ]:
negative_price_frequency = (
    final_data["Negative Price"].mean() * 100
)

negative_price_intervals = final_data["Negative Price"].sum()

print("Negative-price intervals:", negative_price_intervals)
print(f"Negative-price frequency: {negative_price_frequency:.2f}%")

## 4. Market conditions during negative vs non-negative prices

This first comparison looks at average demand, renewable generation, thermal generation and battery behaviour under the two price states.

In [ ]:
negative_price_comparison = (
    final_data
    .groupby("Negative Price")[
        [
            "RRP",
            "TOTALDEMAND",
            "Renewable MW",
            "Thermal MW",
            "Battery Storage"
        ]
    ]
    .mean()
)

negative_price_comparison

## 5. Demand and renewable output

Demand and renewable output are divided into three quantile-based groups (Low / Medium / High).  
This is an analytical grouping, not an AEMO market classification.

In [ ]:
demand_negative_rate = (
    final_data
    .groupby("Demand Group", observed=True)["Negative Price"]
    .mean()
    .mul(100)
)

renewable_negative_rate = (
    final_data
    .groupby("Renewable Group", observed=True)["Negative Price"]
    .mean()
    .mul(100)
)

print("Demand groups:")
print(demand_negative_rate)

print("\nRenewable groups:")
print(renewable_negative_rate)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

demand_negative_rate.plot(
    kind="bar",
    ax=axes[0]
)
axes[0].set_title("Negative Price Frequency by Demand Level")
axes[0].set_xlabel("Demand Group")
axes[0].set_ylabel("Negative Price Frequency (%)")
axes[0].tick_params(axis="x", rotation=0)

renewable_negative_rate.plot(
    kind="bar",
    ax=axes[1]
)
axes[1].set_title("Negative Price Frequency by Renewable Output")
axes[1].set_xlabel("Renewable Group")
axes[1].set_ylabel("Negative Price Frequency (%)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 6. Demand × renewable interaction

The combination of low demand and high renewable output is the strongest negative-price condition observed in the dataset.

In [ ]:
interaction_table = (
    final_data
    .groupby(
        ["Demand Group", "Renewable Group"],
        observed=True
    )["Negative Price"]
    .mean()
    .mul(100)
    .unstack()
)

interaction_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

image = ax.imshow(interaction_table)

ax.set_xticks(range(len(interaction_table.columns)))
ax.set_xticklabels(interaction_table.columns)
ax.set_yticks(range(len(interaction_table.index)))
ax.set_yticklabels(interaction_table.index)

ax.set_xlabel("Renewable Output Group")
ax.set_ylabel("Demand Group")
ax.set_title(
    "Negative Price Frequency by Demand and Renewable Output"
)

for i in range(len(interaction_table.index)):
    for j in range(len(interaction_table.columns)):
        value = interaction_table.iloc[i, j]
        ax.text(
            j, i,
            f"{value:.2f}%",
            ha="center",
            va="center"
        )

plt.colorbar(
    image,
    ax=ax,
    label="Negative Price Frequency (%)"
)

plt.tight_layout()
plt.show()

## 7. Solar and wind analysis

In [ ]:
renewable_comparison = (
    final_data
    .groupby("Negative Price")[
        ["Solar PV", "Wind", "Hydro"]
    ]
    .mean()
)

pct_change = (
    (
        renewable_comparison.loc[True]
        - renewable_comparison.loc[False]
    )
    / renewable_comparison.loc[False]
    * 100
)

print("Average generation:")
display(renewable_comparison)

print("\n% difference during negative-price intervals:")
display(pct_change)

In [ ]:
solar_demand_table = (
    final_data
    .groupby(
        ["Demand Group", "Solar Group"],
        observed=True
    )["Negative Price"]
    .mean()
    .mul(100)
    .unstack()
)

wind_demand_table = (
    final_data
    .groupby(
        ["Demand Group", "Wind Group"],
        observed=True
    )["Negative Price"]
    .mean()
    .mul(100)
    .unstack()
)

print("Solar × Demand")
display(solar_demand_table)

print("Wind × Demand")
display(wind_demand_table)

## 8. Intraday patterns

AEMO SETTLEMENTDATE represents the end of each five-minute interval, so five minutes are subtracted before deriving the analytical hour.

In [ ]:
final_data["Interval Hour"] = (
    final_data["SETTLEMENTDATE"]
    - pd.Timedelta(minutes=5)
).dt.hour

hourly_negative = (
    final_data
    .groupby("Interval Hour")["Negative Price"]
    .mean()
    .mul(100)
)

hourly_solar = (
    final_data
    .groupby("Interval Hour")["Solar PV"]
    .mean()
)

print("Negative price frequency by hour:")
display(hourly_negative)

print("Average solar output by hour:")
display(hourly_solar)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

hourly_negative.plot(ax=axes[0])
axes[0].set_title("Negative Price Frequency by Hour")
axes[0].set_xlabel("Interval Hour")
axes[0].set_ylabel("Negative Price Frequency (%)")

hourly_solar.plot(ax=axes[1])
axes[1].set_title("Average Solar PV by Hour")
axes[1].set_xlabel("Interval Hour")
axes[1].set_ylabel("Average Solar PV (MW)")

plt.tight_layout()
plt.show()

## 9. Extreme price risk

In [ ]:
extreme_summary = {
    "P95 RRP": final_data["RRP"].quantile(0.95),
    "P99 RRP": final_data["RRP"].quantile(0.99),
    "Maximum RRP": final_data["RRP"].max()
}

pd.Series(extreme_summary)

In [ ]:
top_20_prices = (
    final_data[
        ["SETTLEMENTDATE", "RRP", "TOTALDEMAND"]
    ]
    .nlargest(20, "RRP")
)

top_20_prices

In [ ]:
p95_threshold = final_data["RRP"].quantile(0.95)
p99_threshold = final_data["RRP"].quantile(0.99)

final_data["P95+"] = final_data["RRP"] >= p95_threshold
final_data["P99+"] = final_data["RRP"] >= p99_threshold

p95_by_hour = (
    final_data
    .groupby("Interval Hour")["P95+"]
    .mean()
    .mul(100)
)

p99_by_hour = (
    final_data
    .groupby("Interval Hour")["P99+"]
    .mean()
    .mul(100)
)

fig, ax = plt.subplots(figsize=(8, 4))

p95_by_hour.plot(ax=ax, label="P95+")
p99_by_hour.plot(ax=ax, label="P99+")

ax.set_title("High-Price Frequency by Hour")
ax.set_xlabel("Interval Hour")
ax.set_ylabel("Frequency (%)")
ax.legend()

plt.tight_layout()
plt.show()

## 10. Case study: 8 July 2026 extreme price event

The highest observed RRP occurred at 19:40 on 8 July 2026.  
The surrounding five-minute intervals are inspected below to distinguish a genuine market episode from an isolated bad row.

In [ ]:
check_spike = final_data[
    (final_data["SETTLEMENTDATE"] >= "2026-07-08 19:00:00") &
    (final_data["SETTLEMENTDATE"] <= "2026-07-08 20:30:00")
][
    [
        "SETTLEMENTDATE",
        "RRP",
        "TOTALDEMAND",
        "Coal",
        "Gas Turbine",
        "Hydro",
        "Solar PV",
        "Wind",
        "Battery Storage"
    ]
]

check_spike

The event shows a clear escalation across adjacent intervals rather than a single isolated value.  
At the peak, demand was high, solar output was near zero, wind output was very low, and flexible generation/storage output was elevated.

These conditions are consistent with a tight evening supply-demand balance. They should not be interpreted as proof that low renewable output alone caused the price spike, because constraints, outages, interconnector conditions and bidding behaviour can also influence RRP.

## 11. Weekday/weekend and monthly patterns

In [ ]:
interval_start = (
    final_data["SETTLEMENTDATE"]
    - pd.Timedelta(minutes=5)
)

final_data["Day Type"] = (
    interval_start.dt.dayofweek
    .map(lambda x: "Weekday" if x < 5 else "Weekend")
)

final_data["Month"] = interval_start.dt.to_period("M").astype(str)

day_type_frequency = (
    final_data
    .groupby("Day Type")["Negative Price"]
    .mean()
    .mul(100)
)

monthly_frequency = (
    final_data
    .groupby("Month")["Negative Price"]
    .mean()
    .mul(100)
)

print("Weekday vs weekend:")
display(day_type_frequency)

print("Monthly negative-price frequency:")
display(monthly_frequency)

## 12. Key findings

- Negative wholesale prices occurred in roughly one quarter of five-minute intervals during the 12-month observation window.
- Negative prices were strongly associated with lower demand and higher renewable generation.
- The highest negative-price concentration occurred when **low demand and high renewable output coincided**.
- Solar showed a particularly strong midday association with negative pricing; wind also remained important across broader demand conditions.
- High-price risk concentrated in the late-afternoon/evening period.
- The maximum observed RRP occurred at **19:40 on 8 July 2026**, during high demand, near-zero solar output and very low wind output.
- Negative prices were more frequent on weekends than weekdays.
- Monthly variation is descriptive of this 12-month window and should not be treated as evidence of a long-term trend.

## Interpretation note

The project identifies associations, not causal relationships. Wholesale electricity prices are also influenced by network constraints, interconnector flows, generator availability, bidding behaviour, storage activity and other operational conditions.